# Предсказание следующего продукта с помощью GRU

В этом ноутбуке решается задача предсказания следующего продукта, с которым взаимодействует клиент банка, на основе истории его действий

Используется гибридная нейронная архитектура, объединяющая:

- **GRU (Gated Recurrent Unit)** - анализирует последовательность последних 512 событий с учётом их порядка и временных характеристик
- **Attention pooling** - выделяет наиболее информативные скрытые состояния GRU и формирует представление последовательности
- **MLP (Multilayer Perceptron)** - обрабатывает агрегированные признаки всей доступной истории клиента и его взаимодействий с продуктами

Полученные представления объединяются для предсказания одного из 32 целевых продуктов.

**Результаты на Kaggle Public Leaderboard:**

| Модель | Accuracy |
|---|---:|
| GRU | **0.41203** |

Лучший результат получен самостоятельной GRU

## 1. Данные и настройки

Подключим библиотеки, зададим параметры модели и обучения, определим устройство (CPU/GPU)

Загрузим метаданные обучающей и тестовой выборок, а также словарь событий. Подготовим индексы категориальных признаков для embedding-слоёв нейросети

In [ ]:

from pathlib import Path
from time import perf_counter
import hashlib, json, math, os, random, shutil

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
from IPython.display import display


# параметры подготовки данных, архитектуры GRU и обучения
CFG = dict(
    seed=42,
    max_len=512,        # 512 последних событий
    arrow_batch=64,
    batch_size=128,     # примеров в одном батче
    hidden=128,         # скрытое состояние gru
    layers=2,           # 2 слоя
    dropout=0.20,
    epochs=15,
    patience=3,
    lr=0.001,
    weight_decay=0.0001,
    grad_clip=1.0,
    activity_top=16
)

PREPROCESS_VERSION = 'gru_full_history_v1_1'
SMOKE_TEST = False  # быстрая проверка перел обучением

ON_KAGGLE = Path('/kaggle/input').is_dir()
required = [
    'train.parquet',
    'test.parquet',
    'event_dictionary.parquet',
    'sample_submission.csv'
]

if ON_KAGGLE:
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_DIR = next((path for base in [Path.cwd(), *Path.cwd().parents]
                        for path in [base, base / 'task3']
                        if all((path / 'data' / name).is_file() for name in required)), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Нужны четыре исходных файла соревнования в task3/data/.')
    INPUT_ROOT = PROJECT_DIR

folders = sorted({
    p.parent for p in INPUT_ROOT.rglob('train.parquet')
    if all((p.parent / name).is_file() for name in required)
})

if len(folders) != 1:
    raise ValueError(f'Нужен один комплект исходных данных; найдено: {folders}')

DATA_DIR = folders[0]


# Сохраняем промежуточные результаты отдельно от исходных данных.
WORK = (
    Path('/kaggle/working') if ON_KAGGLE
    else Path('/tmp')
) / ('gru_smoke' if SMOKE_TEST else 'gru_v1')

WORK.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR = Path('/kaggle/working') if ON_KAGGLE else PROJECT_DIR / 'submits'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)


# Используем GPU при наличии; mixed precision включаем только на CUDA.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'


# фиксируем настройки вычислений и генераторы случайных чисел
torch.set_num_threads(2)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])


print('Устройство:', DEVICE, '| PyTorch:', torch.__version__)
if DEVICE.type == 'cpu' and not SMOKE_TEST:
    print('Для полного обучения включите GPU. Подготовка данных работает на CPU.')


In [ ]:
# вычисляем хеши файлов 
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

source_hashes = {name: sha256_file(DATA_DIR / name) for name in required}


# Загружаем метаданные без длинных последовательностей событий.
train_meta = pq.read_table(
    DATA_DIR / 'train.parquet',
    columns=['example_id', 'client_id', 'target_product']
).to_pandas()

test_meta = pq.read_table(
    DATA_DIR / 'test.parquet',
    columns=['example_id', 'client_id']
).to_pandas()

dictionary = pq.read_table(
    DATA_DIR / 'event_dictionary.parquet'
).to_pandas()


# формируем 32 целевых класса и переводим ответы в индексы от 0 до 31
classes = np.sort(train_meta.target_product.unique()).astype(np.int32)
assert len(classes) == 32

if SMOKE_TEST:
    train_meta, test_meta = train_meta.iloc[:64].copy(), test_meta.iloc[:16].copy()
    CFG.update(max_len=64, batch_size=8, epochs=1)

y = np.searchsorted(classes, train_meta.target_product.to_numpy()).astype(np.int64)

assert train_meta.example_id.is_unique and test_meta.example_id.is_unique
assert dictionary.encoded_event.is_unique


# оставляем первые индексы для embedding:
# события: 0 - пустая позиция, 1 - неизвестное событие;
# продукты: 0 - пустая позиция, 1 - нет продукта, 2 - неизвестный продукт.
event_codes = sorted(dictionary.encoded_event.astype(int))
event_map = {value: i + 2 for i, value in enumerate(event_codes)}

type_map = {
    'view': 1,
    'click': 2,
    'other': 3,
    'session_boundary': 4
}


# таблицы типов событий и признака is_target_event
event_types = np.full(len(event_map) + 2, 5, dtype=np.int64)
event_types[0] = 0

event_targets = np.zeros(len(event_map) + 2, dtype=np.float32)

for row in dictionary.itertuples(index=False):
    event_types[event_map[int(row.encoded_event)]] = type_map.get(
        row.event_type, 5
    )
    event_targets[event_map[int(row.encoded_event)]] = float(
        row.is_target_event
    )

product_map = {int(value): i + 3 for i, value in enumerate(classes)}

print(f'Train: {len(train_meta):,}; test: {len(test_meta):,}; классов: {len(classes)}')


## 2. Разбиение данных

Разделим обучающую выборку с помощью **StratifiedGroupKFold**

Валидационную часть дополнительно разделим на две:

- **ES (Early Stopping)**  для ранней остановки обучения
- **Outer validation** - для итоговой оценки модели на отложенных данных

Разбиение проверено по ранее сохранённым индексам: 79 999 fit / 5 000 ES / 15 001 outer. Контрольные суммы ниже защищают от незаметного изменения индексов между версиями библиотек. Checkpoint успешного GPU-запуска при этой проверке не был доступен, поэтому совпадение индексов само по себе не доказывает точное воспроизведение Public Score 0.41203.


In [ ]:
train_idx, valid_idx = next(
    StratifiedGroupKFold(5, shuffle=True, random_state=42)
    .split(train_meta, y, train_meta.client_id)
)
train_idx, valid_idx = np.sort(train_idx), np.sort(valid_idx)

# для early stopping оставляем только классы, присутствующие в обучении
known_valid = valid_idx[np.isin(y[valid_idx], np.unique(y[train_idx]))]

# выбираем клиентов для ES
clients = np.unique(train_meta.iloc[known_valid].client_id)
es_clients = np.random.default_rng(2068).choice(
    clients,
    min(5000, max(1, len(clients) // 4)),
    replace=False
)

es_idx = known_valid[
    train_meta.iloc[known_valid].client_id.isin(es_clients).to_numpy()
]
outer_idx = valid_idx[
    ~train_meta.iloc[valid_idx].client_id.isin(es_clients).to_numpy()
]

# проверяем полноту разбиения и отсутствие пересечений клиентов
assert np.array_equal(
    np.sort(np.concatenate([train_idx, valid_idx])),
    np.arange(len(train_meta))
)
assert np.array_equal(
    np.sort(np.concatenate([es_idx, outer_idx])),
    valid_idx
)

for a, b in [(train_idx, valid_idx), (es_idx, outer_idx)]:
    assert not set(train_meta.iloc[a].client_id) & set(
        train_meta.iloc[b].client_id
    )

assert len(es_idx) and len(outer_idx)

print('Fit / ES / outer:', len(train_idx), len(es_idx), len(outer_idx))
print(
    'Классов в fit / ES / outer:',
    *(len(np.unique(y[idx])) for idx in [train_idx, es_idx, outer_idx])
)

# Проверяем сохранённое разбиение без внешних файлов.
if not SMOKE_TEST:
    expected_split_hashes = {'train_idx': '2c69ddc930eaaed6bfe78c9ebc82891b68eddf84984a5fa07eccdcf84db2fed6', 'valid_idx': 'adcb3065fa7e1b5f659efff8dae41cf802d81b781dae8959b9c8c9e4842e2e21', 'es_idx': '485069eeb1ff71519991dcd0c35f809b5d157a22990e78234732ad8a0c981750', 'outer_idx': 'a24fb981a6c4f1964b3741cb9845d61295345a1c2f9e9b553748cd1a8057b4c9'}
    for name, rows in [('train_idx', train_idx), ('valid_idx', valid_idx),
                       ('es_idx', es_idx), ('outer_idx', outer_idx)]:
        digest = hashlib.sha256(np.asarray(rows, dtype='<i8').tobytes()).hexdigest()
        assert digest == expected_split_hashes[name], (
            'Разбиение отличается от проверенного: ' + name
            + '. Проверьте данные и версию scikit-learn; индексы не подменяются автоматически.'
        )


## 3. Последовательность и признаки всей истории

Подготовим два вида признаков для нейросети:

- **Последовательные** - коды событий и продуктов, типы действий и временные характеристики последних 512 событий для GRU
- **Агрегированные** - общая активность клиента и статистики взаимодействий с каждым продуктом за всю доступную историю для MLP

Также учитываем давность действий, недавнюю активность и повторяемость интереса. Подготовленные массивы сохраняем в кэш, чтобы не пересчитывать признаки при повторных запусках

In [ ]:
# Признаки всей истории клиента (30)
GLOBAL_NAMES = [
    # Размер истории и разнообразие продуктов
    'log_length', 'log_duration_hours', 'log_unique_products', 'known_product_rate',

    # Доли различных типов действий
    'view_rate', 'click_rate', 'target_rate', 'other_rate', 'boundary_rate',

    # Сессии и распределение активности по времени
    'log_sessions', 'last_session_fraction', 'log_active_days', 'log_events_per_day',
    'log_gap_mean', 'log_gap_std', 'log_gap_median', 'log_gap_max', 'missing_time_rate',

    # Повторяемость и концентрация интереса к продуктам
    'click_repeat', 'target_repeat',
    'click_entropy', 'target_entropy',
    'click_concentration', 'target_concentration',

    # Активность в последних 20 и 100 событиях
    'last20_click_rate', 'last20_target_rate',
    'last100_click_rate', 'last100_target_rate',

    # Наличие целевых действий и доля непросмотровых событий
    'has_target', 'nonview_rate'
]


# Признаки каждого из 32 целевых продуктов (27 на продукт)
ACTION_NAMES = [
    'log_count',        # количество действий
    'share',            # доля среди действий данного типа
    'last20_share',     # доля в последних 20 событиях
    'last100_share',    # доля в последних 100 событиях
    'decayed_share',    # доля с уменьшением веса старых действий
    'log_age_hours',    # время с последнего действия
    'time_missing',     # отсутствует временная метка
    'seen',             # было ли действие с этим продуктом
    'log_sessions',     # число сессий с действиями
    'growth'            # изменение интереса относительно всей истории
]

# первые 20 признаков: по 10 отдельно для click и target
PRODUCT_NAMES = [
    f'{kind}_{name}'
    for kind in ['click', 'target']
    for name in ACTION_NAMES
] + [
    # Общая информация о взаимодействии с продуктом
    'log_age_any_hours',
    'time_missing_any',
    'last_position',
    'seen_any',

    # Совпадение с последними продуктами в истории
    'last_product',
    'last_click_product',
    'last_target_product'
]

def history_arrays(times, events, products):
    n, C = len(times), len(classes)
    kinds = event_types[events]
    click, target = kinds == 2, event_targets[events] > 0
    known, valid_time = np.isfinite(products), np.isfinite(times)

    # проверяем порядок событий: переставлять их нельзя, так как GRU учитывает хронологию
    assert np.all(np.diff(times[valid_time]) >= 0), 'Нарушен хронологический порядок'

    end = times[valid_time][-1] if valid_time.any() else np.nan
    gap = np.full(n, np.nan)
    known_time_idx = np.flatnonzero(valid_time)
    gap[known_time_idx[1:]] = np.diff(times[valid_time])
    age = np.where(valid_time, np.maximum(end - times, 0), 0)

    # разделяем историю на сессии по перерывам более 30 минут
    sessions = np.cumsum(np.nan_to_num(gap, nan=0) > 1800)
    last_session = sessions == sessions[-1] if n else np.zeros(0, bool)
    duration = np.ptp(times[valid_time]) / 3600 if valid_time.any() else 0
    days = len(np.unique(np.floor(times[valid_time] / 86400)))

    good_gaps = gap[np.isfinite(gap)] / 60
    gap_stats = [np.mean(good_gaps), np.std(good_gaps), np.median(good_gaps), np.max(good_gaps)] if len(good_gaps) else [0]*4

    # насколько действия распределены между продуктами
    concentration = []
    for mask in [click, target]:
        _, counts = np.unique(products[mask & known], return_counts=True)
        shares = counts / max(counts.sum(), 1)
        concentration.append((1 - len(counts)/max(counts.sum(), 1) if len(counts) else 0,
                              -(shares*np.log(shares)).sum(), shares.max() if len(shares) else 0))

    # общие признаки активности клиента за всю доступную историю
    g = [np.log1p(n), np.log1p(duration), np.log1p(len(np.unique(products[known]))), known.mean() if n else 0]
    g += [mask.mean() if n else 0 for mask in [kinds==1, click, target, kinds==3, kinds==4]]
    g += [np.log1p(sessions[-1]+1) if n else 0, last_session.mean() if n else 0,
          np.log1p(days), np.log1p(n/max(days,1)), *np.log1p(gap_stats), 1-valid_time.mean() if n else 0]
    g += [concentration[0][0], concentration[1][0], concentration[0][1], concentration[1][1], concentration[0][2], concentration[1][2]]
    g += [mask[-w:].mean() if n else 0 for w in [20,100] for mask in [click,target]]
    g += [float(target.any()), (kinds!=1).mean() if n else 0]
    assert len(g) == len(GLOBAL_NAMES)

    # сопоставляем продукты из истории с 32 возможными целевыми классами
    candidate = np.searchsorted(classes, np.nan_to_num(products, nan=-1))
    match = (candidate < C) & known
    match &= classes[np.minimum(candidate, C-1)] == products
    candidate = np.where(match, candidate, -1)

    def counts(mask, weights=None):
        ok = mask & match
        return np.bincount(candidate[ok], weights=None if weights is None else weights[ok], minlength=C).astype(np.float32)

    # для каждого продукта считаем частоты, давность, сессии и изменение интереса
    panel = []
    for mask in [click, target]:
        total = counts(mask)
        share = total / max(mask.sum(), 1)
        recent = []

        for w in [20,100]:
            tail = np.arange(n) >= max(0,n-w)
            recent.append(counts(mask & tail) / max((mask & tail).sum(), 1))

        # более старые действия получают меньший вес
        weight = np.where(valid_time, np.exp2(-age/86400), 0)
        decayed = counts(mask, weight) / max(weight[mask].sum(), 1e-8)

        ages, missing, seen, ns = np.zeros(C), np.ones(C), total>0, np.zeros(C)
        for j in np.flatnonzero(seen):
            idx = np.flatnonzero(mask & (candidate==j))
            last = idx[-1]
            if valid_time[last]: ages[j], missing[j] = np.log1p(age[last]/3600), 0
            ns[j] = np.log1p(len(np.unique(sessions[idx])))

        panel += [np.log1p(total), share, *recent, decayed, ages, missing, seen, ns, recent[1]-share]

    any_age, missing, position, seen = np.zeros(C), np.ones(C), np.zeros(C), np.zeros(C)
    for j in np.unique(candidate[match]):
        idx = np.flatnonzero(candidate==j)[-1]
        seen[j], position[j] = 1, (n-1-idx)/max(n,1)
        if valid_time[idx]: any_age[j], missing[j] = np.log1p(age[idx]/3600), 0

    # отмечаем последние продукты в истории и в действиях разных типов
    last_products = []
    for mask in [known, click & known, target & known]:
        idx = np.flatnonzero(mask)
        last_products.append((classes == products[idx[-1]]).astype(np.float32) if len(idx) else np.zeros(C))

    panel = np.column_stack(panel + [any_age, missing, position, seen, *last_products]).astype(np.float32)
    assert panel.shape == (C, len(PRODUCT_NAMES))

    # профиль непросмотровых действий клиента
    activity = np.bincount(events[kinds!=1], minlength=len(event_types)).astype(np.uint16)

    # 6 числовых признаков для каждого события последовательности
    numerics = np.column_stack([np.log1p(np.nan_to_num(gap, nan=0))/16,
        np.log1p(age)/16, valid_time, event_targets[events],
        np.where(valid_time, np.sin(2*np.pi*np.nan_to_num(times)/86400), 0),
        np.where(valid_time, np.cos(2*np.pi*np.nan_to_num(times)/86400), 0)]).astype(np.float16)

    return np.asarray(g,np.float32), panel, activity, numerics, end


In [ ]:
cache_signature = hashlib.sha256(json.dumps(dict(version=PREPROCESS_VERSION,
    sources=source_hashes, length=CFG['max_len'], smoke=SMOKE_TEST), sort_keys=True).encode()).hexdigest()

CACHE = WORK / 'arrays'

# ищем готовый совместимый кэш, чтобы не пересчитывать признаки
cache_candidates = [CACHE] + [p.parent for p in INPUT_ROOT.rglob('gru_cache.json')]
cache_info = None

for candidate in cache_candidates:
    manifest = candidate / 'gru_cache.json'
    if manifest.exists():
        info = json.loads(manifest.read_text())
        if info['signature'] == cache_signature and all((candidate / f'{s}_{k}.npy').exists()
            for s in ['train','test'] for k in ['tokens','time','length','global','product','activity','end']):
            CACHE, cache_info = candidate, info
            product_map = {int(k):v for k,v in info['product_map'].items()}
            break


def prepare_file(path, size, prefix):
    # создаём массивы на диске через memmap
    shapes = dict(tokens=((size,CFG['max_len'],3),'uint16'), time=((size,CFG['max_len'],6),'float16'),
        length=((size,),'uint16'), global_=((size,len(GLOBAL_NAMES)),'float32'),
        product=((size,len(classes),len(PRODUCT_NAMES)),'float32'),
        activity=((size,len(event_types)),'uint16'), end=((size,),'float64'))

    arrays = {key.rstrip('_'): np.lib.format.open_memmap(CACHE / f'{prefix}_{key.rstrip("_")}.npy',
                mode='w+',dtype=dtype,shape=shape) for key,(shape,dtype) in shapes.items()}

    offset = 0

    # читаем длинные истории небольшими пакетами из Parquet
    for batch in tqdm(pq.ParquetFile(path).iter_batches(batch_size=CFG['arrow_batch'], columns=['events'], use_threads=False),
                      total=math.ceil(size/CFG['arrow_batch']), desc=prefix):
        lists = batch.column('events'); values = lists.values
        offsets = lists.offsets.to_numpy()

        # переводим миллисекунды в секунды
        ts = values.field('timestamp').to_numpy(zero_copy_only=False).astype(np.float64) * 0.001
        raw_event = values.field('encoded_event').to_numpy(zero_copy_only=False)
        raw_product = values.field('encoded_product').to_numpy(zero_copy_only=False).astype(np.float64)

        unique_events, inverse = np.unique(raw_event, return_inverse=True)
        event_ids = np.asarray([event_map.get(int(v),1) if np.isfinite(v) else 1 for v in unique_events])[inverse]

        # добавляем встреченные продукты в словарь embedding-индексов
        for value in np.unique(raw_product[np.isfinite(raw_product)]):
            product_map.setdefault(int(value), len(product_map)+3)

        product_ids = np.ones(len(raw_product),dtype=np.uint16)
        for value in np.unique(raw_product[np.isfinite(raw_product)]):
            product_ids[raw_product==value] = product_map[int(value)]

        assert max(product_map.values()) < 65536

        for i in range(min(len(batch), size-offset)):
            start, stop = offsets[i:i+2]
            t,e,p = ts[start:stop],event_ids[start:stop],raw_product[start:stop]
            g, panel, activity, numbers, end = history_arrays(t,e,p)

            row, length = offset+i, min(len(e),CFG['max_len'])
            arrays['global'][row],arrays['product'][row],arrays['activity'][row],arrays['end'][row] = g,panel,activity,end

            arrays['tokens'][row] = 0; arrays['time'][row] = 0
            arrays['length'][row] = max(length,1)

            # GRU получает последние max_len событий, всю историю
            if length:
                arrays['tokens'][row,:length] = np.column_stack([e[-length:],product_ids[start:stop][-length:],event_types[e[-length:]]])
                arrays['time'][row,:length] = numbers[-length:]
            else:
                # Для пустой истории создаём одно нейтральное событие
                arrays['tokens'][row,0] = [1,1,5]

        offset += min(len(batch),size-offset)
        if offset == size: break

    assert offset == size
    for array in arrays.values(): array.flush()
    del arrays


# при отсутствии совместимого кэша рассчитываем признаки заново
if cache_info is None:
    CACHE.mkdir(parents=True,exist_ok=True)
    prepare_file(DATA_DIR/'train.parquet',len(train_meta),'train')
    prepare_file(DATA_DIR/'test.parquet',len(test_meta),'test')

    cache_info = dict(signature=cache_signature, product_map=product_map, classes=classes.tolist(),
                      global_names=GLOBAL_NAMES, product_names=PRODUCT_NAMES)

    (CACHE/'gru_cache.json').write_text(json.dumps(cache_info,ensure_ascii=False,indent=2))

# открываем массивы без полной загрузки в оперативную память
arrays = {split:{key:np.load(CACHE/f'{split}_{key}.npy',mmap_mode='r')
                 for key in ['tokens','time','length','global','product','activity','end']} for split in ['train','test']}

assert cache_info['classes'] == classes.tolist()
assert cache_info['global_names'] == GLOBAL_NAMES and cache_info['product_names'] == PRODUCT_NAMES

print('Продуктовых кодов:',len(product_map), '| кеш, ГБ:',round(sum(p.stat().st_size for p in CACHE.glob('*.npy'))/1024**3,2))


## 4. Подготовка входов модели

Выберем 16 наиболее частых непросмотровых действий и добавим их к общим признакам клиента

Нормализуем числовые признаки, рассчитывая средние значения и стандартные отклонения только на обучающей части (fit), чтобы избежать утечки данных.

Затем создадим `Dataset` и `DataLoader` для формирования батчей с последовательностями событий и агрегированными признаками.

In [ ]:
# выбираем самые частые непросмотровые действия по обучающей выборке
profile_sum = np.zeros(len(event_types), np.float64)
for start in range(0, len(train_idx), 2048):
    block = np.asarray(arrays['train']['activity'][train_idx[start:start+2048]], np.float32)
    profile_sum += (block / np.maximum(block.sum(axis=1, keepdims=True), 1)).sum(axis=0)

activity_cols = np.argsort(-profile_sum, kind='stable')[:CFG['activity_top']]
activity_cols = activity_cols[profile_sum[activity_cols] > 0]


def global_inputs(block):
    activity = np.asarray(block['activity'], np.float32)
    count = activity.sum(axis=1, keepdims=True)

    selected = activity[:, activity_cols] / np.maximum(count, 1)
    return np.column_stack([
        block['global'],
        selected,
        np.where(count[:, 0] > 0, 1 - selected.sum(axis=1), 0),
        count[:, 0] > 0
    ]).astype(np.float32)


# собираем общие признаки (для train и test)
global_values = {}
for s in ['train', 'test']:
    parts = []
    for start in range(0, len(arrays[s]['global']), 2048):
        parts.append(global_inputs({
            key: arrays[s][key][start:start+2048]
            for key in ['global', 'activity']
        }))
    global_values[s] = np.concatenate(parts)
    del parts


# среднее и стандартное отклонение считаем только на fit
g_mean = global_values['train'][train_idx].mean(axis=0)
g_std = np.maximum(global_values['train'][train_idx].std(axis=0), 0.05)


# нормализация признаков 
p_sum = np.zeros(len(PRODUCT_NAMES))
p_sq = np.zeros(len(PRODUCT_NAMES))
n_product_rows = 0

for start in range(0, len(train_idx), 1024):
    block = np.asarray(
        arrays['train']['product'][train_idx[start:start+1024]],
        np.float64
    ).reshape(-1, len(PRODUCT_NAMES))

    p_sum += block.sum(axis=0)
    p_sq += (block * block).sum(axis=0)
    n_product_rows += len(block)

p_mean = (p_sum / n_product_rows).astype(np.float32)
p_std = np.maximum(
    np.sqrt(np.maximum(p_sq / n_product_rows - p_mean.astype(float)**2, 0)),
    0.05
).astype(np.float32)


# приводим общие признаки к единому масштабу и ограничиваем выбросы
for s in global_values:
    global_values[s] = np.clip(
        (global_values[s] - g_mean) / g_std, -10, 10
    ).astype(np.float32)

class_tokens = np.asarray([product_map[int(c)] for c in classes])

print('Общих признаков:', global_values['train'].shape[1],
      '| на продукт:', len(PRODUCT_NAMES))


In [ ]:
class HistoryDataset(Dataset):
    def __init__(self, split, rows):
        self.split, self.rows = split, np.asarray(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        data = arrays[self.split]
        length = int(data['length'][row])

        # нормализуем признаки 
        product = np.clip(
            (np.asarray(data['product'][row]) - p_mean) / p_std,
            -10, 10
        ).astype(np.float32)

        # Возвращаем последовательность, агрегаты, ответ и индекс клиента
        return (
            np.array(data['tokens'][row, :length], dtype=np.int64),
            np.array(data['time'][row, :length], dtype=np.float32),
            global_values[self.split][row],
            product,
            int(y[row]) if self.split == 'train' else -1,
            int(row)
        )


def collate_histories(samples):
    lengths = torch.tensor([len(s[0]) for s in samples], dtype=torch.long)
    tokens = torch.zeros(len(samples), int(lengths.max()), 3, dtype=torch.long)
    time = torch.zeros(len(samples), int(lengths.max()), 6)

    for i, s in enumerate(samples):
        tokens[i, :lengths[i]] = torch.from_numpy(s[0])
        time[i, :lengths[i]] = torch.from_numpy(s[1])

    # собираем все входы нейросети в один батч
    return dict(
        tokens=tokens,
        time=time,
        lengths=lengths,
        global_=torch.from_numpy(np.stack([s[2] for s in samples])),
        product=torch.from_numpy(np.stack([s[3] for s in samples])),
        label=torch.tensor([s[4] for s in samples]),
        row=torch.tensor([s[5] for s in samples])
    )


def loader(split, rows, shuffle=False, seed=42):
    # DataLoader формирует батчи и при необходимости перемешивает примеры
    return DataLoader(
        HistoryDataset(split, rows),
        batch_size=CFG['batch_size'],
        shuffle=shuffle,
        collate_fn=collate_histories,
        num_workers=0,
        pin_memory=AMP,
        generator=torch.Generator().manual_seed(seed)
    )


## 5. Архитектура GRU

Модель объединяет последовательные и агрегированные признаки клиента

Последние 512 событий преобразуются с помощью **embeddings** и обрабатываются двухслойной **GRU** со скрытым состоянием размерности 128. Для представления последовательности используются последнее скрытое состояние и **attention pooling**, который выделяет наиболее информативные события

Параллельно две **MLP** обрабатывают признаки взаимодействий с отдельными продуктами и общие характеристики всей истории клиента. Полученные представления объединяются, после чего модель вычисляет оценки для 32 целевых продуктов

In [ ]:
class ProductGRU(nn.Module):
    def __init__(self):
        super().__init__()

        # embedding-слои превращают категориальные индексы в обучаемые векторы
        self.event_emb = nn.Embedding(len(event_types), 48, padding_idx=0)
        self.product_emb = nn.Embedding(max(product_map.values()) + 1, 32, padding_idx=0)
        self.type_emb = nn.Embedding(6, 8, padding_idx=0)

        self.input = nn.Sequential(
            nn.Linear(94, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        # обрабатываем события в хронологическом порядке
        self.gru = nn.GRU(
            96, CFG['hidden'],
            num_layers=CFG['layers'],
            batch_first=True,
            dropout=0.15 if CFG['layers'] > 1 else 0
        )

        self.attention = nn.Linear(CFG['hidden'], 1)

        # MLP для 27 агрегированных признаков каждого продукта
        self.product_net = nn.Sequential(
            nn.Linear(len(PRODUCT_NAMES), 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(CFG['dropout']),
            nn.Linear(64, 32)
        )

        # MLP для общих признаков истории и усреднённых признаков продуктов
        self.global_net = nn.Sequential(
            nn.Linear(global_values['train'].shape[1] + 32, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(CFG['dropout'])
        )

        # объединяем последнее состояние GRU, attention pooling и глобальные признаки
        self.fusion = nn.Sequential(
            nn.Linear(CFG['hidden'] * 2 + 96, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(CFG['dropout'])
        )

        self.query = nn.Linear(128, 32)
        self.head = nn.Linear(128, len(classes))

        # индексы 32 целевых продуктов хранятся вместе с моделью,
        self.register_buffer(
            'class_tokens',
            torch.tensor(class_tokens, dtype=torch.long)
        )

    def forward(self, batch):
        token = batch['tokens']
        length = batch['lengths']

        # три embedding-вектора + шесть временных признаков
        x = torch.cat([
            self.event_emb(token[:, :, 0]),
            self.product_emb(token[:, :, 1]),
            self.type_emb(token[:, :, 2]),
            batch['time']
        ], dim=-1)
        x = self.input(x)

        # упаковываем последовательности, чтобы GRU
        # не обрабатывала добавленные при padding пустые позиции
        packed = pack_padded_sequence(
            x, length.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        out, hidden = self.gru(packed)

        # возвращаем скрытые состояния к форме обычного батча
        out, _ = pad_packed_sequence(
            out,
            batch_first=True,
            total_length=token.shape[1]
        )

        # attention pooling
        mask = (
            torch.arange(token.shape[1], device=token.device)[None, :]
            < length.to(token.device)[:, None]
        )
        weight = torch.softmax(
            self.attention(out).squeeze(-1).float().masked_fill(~mask, -1e9),
            dim=1
        )

        # взвешенная сумма состояний GRU - представление последовательности
        pooled = (out * weight.unsqueeze(-1)).sum(dim=1)

        # обрабатываем признаки через MLP
        product = self.product_net(batch['product'])

        # Усредняем представления продуктов и объединяем
        global_ = self.global_net(
            torch.cat([batch['global_'], product.mean(dim=1)], dim=-1)
        )

        # последнее состояние GRU + attention pooling + глобальные признаки
        context = self.fusion(
            torch.cat([hidden[-1], pooled, global_], dim=-1)
        )

        candidates = product + self.product_emb(self.class_tokens)[None, :, :]

        return (
            self.head(context)
            + (candidates * self.query(context)[:, None, :]).sum(-1)
              / math.sqrt(32)
        )


# создаём модель и выводим количество обучаемых параметров
seed_everything(CFG['seed'])
model = ProductGRU().to(DEVICE)

print(f'Параметров модели: {sum(p.numel() for p in model.parameters()):,}')


## 6. Обучение GRU

Обучим модель с помощью **AdamW** и функции потерь **CrossEntropyLoss**

После каждой эпохи оцениваем Accuracy на выборке ES. Если качество не улучшается три эпохи подряд, обучение останавливается (**early stopping**)

Сохраняем два checkpoint: `best.pt` с лучшими весами модели и `last.pt` для продолжения прерванного обучения. Отложенная выборка outer на этом этапе не используется

In [ ]:
def to_device(batch):
    return {
        key: (value if key in ['lengths', 'row'] else value.to(DEVICE, non_blocking=AMP))
        for key, value in batch.items()
    }


# получаем вероятности классов без вычисления градиентов
@torch.no_grad()
def predict(rows, split='train'):
    model.eval()
    result = []
    actual_rows = []

    for batch in loader(split, rows):
        actual_rows.append(batch['row'].numpy())

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            logits = model(to_device(batch))

        prob = logits.float().softmax(-1).cpu().numpy()
        assert np.isfinite(prob).all()
        result.append(prob)

    assert np.array_equal(np.concatenate(actual_rows), rows)
    return np.concatenate(result)


# сохраняем checkpoint через временный файл, чтобы избежать повреждения при прерывании
def atomic_save(value, path):
    temporary = path.with_suffix('.tmp')
    torch.save(value, temporary)
    os.replace(temporary, path)


# сохраняем параметры предобработки и разбиения для воспроизводимости
preprocessing = dict(
    g_mean=g_mean, g_std=g_std,
    p_mean=p_mean, p_std=p_std,
    activity_cols=activity_cols,
    event_map=event_map, product_map=product_map,
    class_tokens=class_tokens, classes=classes,
    global_names=GLOBAL_NAMES, product_names=PRODUCT_NAMES,
    source_hashes=source_hashes,
    train_idx=train_idx, valid_idx=valid_idx,
    es_idx=es_idx, outer_idx=outer_idx,
    versions=dict(
        torch=str(torch.__version__),
        numpy=np.__version__,
        pandas=pd.__version__
    )
)

run_signature = hashlib.sha256(
    json.dumps(
        dict(
            cache=cache_signature,
            config=CFG,
            train_idx=train_idx.tolist(),
            es_idx=es_idx.tolist(),
            activity_cols=activity_cols.tolist()
        ),
        sort_keys=True
    ).encode()
).hexdigest()


# AdamW, уменьшение learning rate и CrossEntropyLoss
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay']
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG['epochs'],
    eta_min=CFG['lr'] * 0.05
)

scaler = torch.amp.GradScaler('cuda', enabled=AMP)
criterion = nn.CrossEntropyLoss(label_smoothing=0.02)

start_epoch, best_accuracy, bad_epochs, logs = 0, -1.0, 0, []
last_path, best_path = WORK / 'last.pt', WORK / 'best.pt'

resume_candidates = [last_path] + sorted(INPUT_ROOT.rglob('last.pt'))

# загружаем только собственные checkpoint, сохранённые этим ноутбуком
for path in resume_candidates:
    if not path.exists():
        continue

    checkpoint = torch.load(path, map_location='cpu', weights_only=False)

    if checkpoint.get('signature') != run_signature:
        continue

    source_best = path.with_name('best.pt')
    if not source_best.exists():
        raise FileNotFoundError('Для продолжения нужны last.pt и best.pt')

    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    scaler.load_state_dict(checkpoint['scaler'])

    start_epoch = checkpoint['epoch'] + 1
    best_accuracy = checkpoint['best_accuracy']
    bad_epochs = checkpoint['bad_epochs']
    logs = checkpoint['logs']

    if source_best.resolve() != best_path.resolve():
        shutil.copyfile(source_best, best_path)

    print('Продолжаем с эпохи', start_epoch + 1)
    del checkpoint
    break


if DEVICE.type != 'cuda' and not SMOKE_TEST:
    raise RuntimeError('Полное обучение на CPU отключено. Включите GPU в настройках Kaggle.')


# основной цикл обучения
for epoch in range(start_epoch, CFG['epochs']):
    if bad_epochs >= CFG['patience']:
        break

    seed_everything(CFG['seed'] + epoch)
    model.train()

    total_loss = 0.
    total_correct = 0
    total = 0
    started = perf_counter()

    for batch in tqdm(
        loader('train', train_idx, shuffle=True, seed=CFG['seed'] + epoch),
        desc=f'Эпоха {epoch+1}'
    ):
        batch = to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        # Прямой проход и вычисление функции потерь
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            logits = model(batch)
            loss = criterion(logits, batch['label'])

        if not torch.isfinite(loss):
            raise FloatingPointError('Loss не конечен; checkpoint предыдущей эпохи сохранён')

        # обратное распространение ошибки и обновление весов
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            CFG['grad_clip'],
            error_if_nonfinite=not AMP
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(batch['label'])
        total += len(batch['label'])
        total_correct += (logits.argmax(1) == batch['label']).sum().item()

    # оцениваем качество на ES после каждой эпохи
    es_probability = predict(es_idx)
    es_accuracy = accuracy_score(y[es_idx], es_probability.argmax(1))

    es_loss = -np.log(
        np.maximum(
            es_probability[np.arange(len(es_idx)), y[es_idx]],
            1e-12
        )
    ).mean()

    logs.append(dict(
        epoch=epoch + 1,
        train_loss=total_loss / total,
        train_accuracy=total_correct / total,
        es_loss=float(es_loss),
        es_accuracy=float(es_accuracy),
        seconds=perf_counter() - started
    ))

    scheduler.step()

    # сохраняем лучшие веса и сбрасываем счётчик эпох без улучшения
    if es_accuracy > best_accuracy:
        best_accuracy, bad_epochs = es_accuracy, 0

        atomic_save(
            dict(
                model=model.state_dict(),
                epoch=epoch,
                best_accuracy=best_accuracy,
                signature=run_signature,
                config=CFG,
                preprocessing=preprocessing
            ),
            best_path
        )
    else:
        bad_epochs += 1

    # last.pt содержит также оптимизатор и scheduler для продолжения обучения
    atomic_save(
        dict(
            model=model.state_dict(),
            optimizer=optimizer.state_dict(),
            scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(),
            epoch=epoch,
            best_accuracy=best_accuracy,
            bad_epochs=bad_epochs,
            logs=logs,
            signature=run_signature,
            config=CFG,
            preprocessing=preprocessing
        ),
        last_path
    )

    print(pd.Series(logs[-1]).round(5).to_string())

display(pd.DataFrame(logs).round(5))


## 7. Итоговая оценка

Загрузим веса модели с лучшей Accuracy на ES и оценим её качество на отложенной выборке **outer**, которая не использовалась при обучении и выборе эпохи

Дополнительно проверим качество на длинных и более поздних историях клиентов, чтобы оценить устойчивость модели к различиям в данных

In [ ]:
checkpoint = torch.load(best_path,map_location='cpu',weights_only=False)
assert checkpoint['signature'] == run_signature
model.load_state_dict(checkpoint['model']); model.to(DEVICE)
valid_probability = predict(valid_idx)
valid_position = {int(row):i for i,row in enumerate(valid_idx)}
outer_pos = np.asarray([valid_position[int(row)] for row in outer_idx])
lengths = np.expm1(arrays['train']['global'][outer_idx,0])
fit_end = arrays['train']['end'][train_idx]
late_threshold = np.nanquantile(fit_end,0.8)
slices = {'Outer':np.ones(len(outer_idx),bool),'Истории ≥1000 событий':lengths>=999.5,
          'Поздние истории':arrays['train']['end'][outer_idx]>=late_threshold}
results = []
for name,mask in slices.items():
    if mask.any(): results.append(dict(Выборка=name,Историй=int(mask.sum()),
        Accuracy=accuracy_score(y[outer_idx[mask]],valid_probability[outer_pos[mask]].argmax(1))))
most_frequent = np.bincount(y[train_idx],minlength=len(classes)).argmax()
print('Самый частый продукт, outer Accuracy:',round(float((y[outer_idx]==most_frequent).mean()),5))
display(pd.DataFrame(results))
print('Лучшая эпоха:',checkpoint['epoch']+1)
del checkpoint

## 8. Предсказания и сохранение результатов

Получим предсказания GRU для тестовой выборки и выберем продукт с наибольшей вероятностью для каждого клиента

Преобразуем индексы классов обратно в исходные коды продуктов и сохраним результаты в CSV в формате Kaggle, сохранив порядок `example_id` из образца отправки

При локальном запуске CSV сохраняются в `task3/submits/`, на Kaggle — непосредственно в `/kaggle/working/`. Кеш и checkpoints остаются в каталоге `WORK`; существующие CSV не перезаписываются.


In [ ]:
def save_submission(probability,name):
    assert probability.shape==(len(test_meta),len(classes))
    assert np.isfinite(probability).all() and (probability>=0).all()
    assert np.allclose(probability.sum(axis=1),1,atol=1e-5)
    sample = pd.read_csv(DATA_DIR/'sample_submission.csv',dtype={'example_id':str})
    assert list(sample.columns)==['example_id','encoded_product'] and sample.example_id.is_unique
    predicted = pd.DataFrame({'example_id':test_meta.example_id.astype(str),
                              'encoded_product':classes[probability.argmax(axis=1)]})
    assert predicted.example_id.is_unique and set(sample.example_id)==set(predicted.example_id)
    submission = sample[['example_id']].merge(predicted,on='example_id',how='left',validate='one_to_one',sort=False)
    assert submission.example_id.equals(sample.example_id)
    assert len(submission)==len(test_meta) and not submission.isna().any().any()
    assert submission.encoded_product.isin(classes).all()
    path = SUBMISSION_DIR/name
    suffix=1
    while path.exists():
        path=SUBMISSION_DIR/f'{Path(name).stem}_{suffix}.csv'; suffix+=1
    submission.to_csv(path,index=False,mode='x')
    print('Сохранён:',path,'| строк:',len(submission))
    return path

if not SMOKE_TEST:
    test_probability = predict(np.arange(len(test_meta)),split='test')
    np.savez_compressed(WORK/'gru_probabilities.npz',classes=classes,
        valid_ids=train_meta.iloc[valid_idx].example_id.to_numpy(dtype=str),test_ids=test_meta.example_id.to_numpy(dtype=str),
        valid_proba=valid_probability,test_proba=test_probability,
        train_idx=train_idx,valid_idx=valid_idx,es_idx=es_idx,outer_idx=outer_idx,signature=run_signature)
    gru_submission = save_submission(test_probability,'submission_gru.csv')
else:
    print('Smoke test: CSV и test-предсказания для соревнования не создаём.')

## 9. Вывод

В работе разработана гибридная нейросетевая модель, объединяющая **GRU с attention pooling** для обработки последовательности событий и **MLP** для анализа агрегированных признаков клиента и продуктов

Модель достигла **Accuracy 0.41203** на Kaggle Public Leaderboard

Использование последовательных и агрегированных признаков позволяет учитывать как последние действия клиента, так и его долгосрочные интересы. Итоговым решением выбрана самостоятельная GRU

На private **Accuracy** упало до 0.36692